In [1]:
from google.colab import drive
import os
import glob

drive_zip = "/content/drive/MyDrive/speaker-recognition-audio-dataset.zip"
local_zip = "/content/speaker-recognition-audio-dataset.zip"
extract_dir = "/content/dataset"
kaggle_dataset = "vjcalling/speaker-recognition-audio-dataset"

# 1. Check if zip already exists locally or if dataset is already extracted
if not os.path.exists(local_zip) and not os.path.exists(extract_dir):
    # Mount Drive and check for dataset zip
    try:
        if not os.path.exists('/content/drive'):
            drive.mount('/content/drive')
    except Exception as e:
        print(f"Drive mount skipped/failed: {e}")

    if os.path.exists(drive_zip):
        print("Found dataset on Google Drive. Copying zip to local SSD...")
        !rsync -ah --info=progress2 "$drive_zip" "$local_zip"
    else:
        print("Dataset zip not found on Google Drive. Downloading from Kaggle...")
        !pip install -q kaggle
        !kaggle datasets download -d $kaggle_dataset -p /content
        
        # In case the downloaded file is named differently, locate and rename it
        if not os.path.exists(local_zip):
            matches = glob.glob('/content/*speaker-recognition-audio-dataset*.zip') + glob.glob('/content/*speaker*.zip')
            if matches:
                os.rename(matches[0], local_zip)
elif os.path.exists(local_zip):
    print("Zip already exists locally.")
else:
    print("Dataset already extracted.")

# 2. Multi-threaded extraction using 7-Zip (uses all CPU cores)
if os.path.exists(local_zip) and not os.path.exists(extract_dir):
    print("Extracting dataset...")
    !apt-get install -y -qq p7zip-full > /dev/null
    !7z x -mmt=on -o"$extract_dir" "$local_zip" -bso0 -bsp1
    print("Done! Files extracted successfully.")
elif os.path.exists(extract_dir):
    print("Dataset directory already exists.")
else:
    print("Error: Dataset zip file not found.")


Copying zip to local SSD...
          3.91G 100%   86.45MB/s    0:00:43 (xfr#1, to-chk=0/1)
Extracting dataset...
  0M Scan /content                        0%      0% 12 - 50_speakers_audio_data/Speaker0026/Speaker26_012.w                                                              0% 18 - 50_speakers_audio_data/Speaker0026/Speaker26_018.w                                                              1% 25 - 50_speakers_audio_data/Speaker0026/Speaker26_025.w                                                              1% 31 - 50_speakers_audio_data/Speaker0026/Speaker26_031.w                                                              2% 37 - 50_speakers_audio_data/Speaker0026/Speaker26_037.w                                                              2% 42 - 50_speakers_audio_data/Speaker0026/Speaker26_042.w                                                              2% 4        2% 54 - 50_speakers_audio_data/Speaker0027/Speaker27_009.w                                              

In [2]:
import librosa
import glob
import numpy as np

dataset_path = "/content/dataset/50_speakers_audio_data"

# Collect all WAV files recursively
audio_files = glob.glob(dataset_path + "/**/*.wav", recursive=True)

print(f"Files found: {len(audio_files)}")

if len(audio_files) == 0:
  print(" No audio files found. Check path or extension.")
  exit()

Files found: 2511


In [4]:
import soundfile as sf
from concurrent.futures import ThreadPoolExecutor
import numpy as np

def get_audio_info(fpath):
    try:
        # Read metadata only (instant header inspection, no PCM decoding)
        info = sf.info(fpath)
        return info.duration, info.samplerate
    except Exception:
        return None

# Multi-threaded header reading across all audio files
with ThreadPoolExecutor() as executor:
    results = list(executor.map(get_audio_info, audio_files))

# Filter valid results
valid_results = [r for r in results if r is not None]
durations = np.array([r[0] for r in valid_results])
sample_rates = {r[1] for r in valid_results}

print(f"Processed {len(durations)}/{len(audio_files)} files")
print(f"Sample rates seen : {sample_rates}")
print(f"Duration — min : {durations.min():.2f}s")
print(f"Duration — max : {durations.max():.2f}s")
print(f"Duration — mean : {durations.mean():.2f}s")
print(f"Duration — total : {durations.sum()/60:.1f} minutes")


Processed 2511/2511 files
Sample rates seen : {16000, 22050}
Duration — min : 1.12s
Duration — max : 60.32s
Duration — mean : 59.33s
Duration — total : 2483.2 minutes


### Dataset Description

* **Dataset Name**: Speaker Recognition Audio Dataset
* **Source**: Kaggle (`vjcalling/speaker-recognition-audio-dataset`)
* **Number of Audio Files**: 2,511
* **Number of Classes**: 50 Speakers
* **Total Duration**: ~2,482.9 minutes (approximately 41.4 hours)
* **Sampling Rates**: 16,000 Hz and 22,050 Hz
* **File Format**: `.wav`